# NB-06: The DiT Block -- Assembling Self-Attention, Cross-Attention, and FFN

## Learning Objectives
- Trace `DiTBlock.__init__` to see how six sub-modules wire together: `self_attn`, `cross_attn`, `norm1/2/3`, `ffn`, `modulation`, `gate`
- Walk through `DiTBlock.forward()` line-by-line, connecting each operation back to its prior notebook (NB-01 modulate, NB-04 attention, NB-05 adaLN-Zero)
- Count parameters per sub-module and explain why `q`, `k`, `v`, `o`, `ffn.0`, `ffn.2` are LoRA targets

## Prerequisites
- **Prior notebooks:** NB-01 (modulate function), NB-04 (SelfAttention, CrossAttention), NB-05 (adaLN-Zero, GateModule)
- **Assumed concepts:** Residual connections, adaLN-Zero gate=0 identity, 3D RoPE frequency assembly

## Concept Map
- `DiTBlock` assembles Phase 1 primitives into a single composed unit, then is stacked 30x in `WanModel` (NB-08)
- `DiTBlock.modulation` is the per-block learned offset on time conditioning (NB-05 established the pattern)
- `DiTBlock.forward` requires a **full 3-band freqs tensor** — not the simplified single-band shortcut from NB-04

## DiT Block Data Flow

```
DiT Block Data Flow
===================
         x [B, S, dim]
         |
         +---> norm1 ---> modulate(shift_msa, scale_msa) ---> SelfAttention ---> x gate_msa ---> + ---> x'
         |                                                      (RoPE via freqs)                  |
         |<-------------------------------------------------------------------------------------  |
         |                                                                                         |
         +---> norm3 ---> CrossAttention(context) ---------------------------------------------> + ---> x''
         |                                                                                         |
         |<-------------------------------------------------------------------------------------  |
         |                                                                                         |
         +---> norm2 ---> modulate(shift_mlp, scale_mlp) ---> FFN -----------> x gate_mlp ---> + ---> output
         |                                                                                         |
         |<-------------------------------------------------------------------------------------  |
```

Key: self-attention and FFN use adaLN + gated residual (from NB-05). Cross-attention uses a **simple** residual -- no adaLN, no gate.

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):  # search up to 10 levels
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:  # reached filesystem root
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# NB-06 imports -- Source: diffsynth/models/wan_video_dit.py
from diffsynth.models.wan_video_dit import (
    DiTBlock, GateModule, modulate, precompute_freqs_cis_3d
)
import torch
import torch.nn as nn

print("Setup complete.")

## 1. DiTBlock.__init__ -- Sub-Module Wiring

`DiTBlock` assembles six sub-modules from the Phase 1 primitives into a single composed unit:

| Sub-module | Type | Source Notebook |
|-----------|------|----------------|
| `self_attn` | `SelfAttention` | NB-04 |
| `cross_attn` | `CrossAttention` | NB-04 |
| `norm1`, `norm2` | `LayerNorm` (no affine) | -- |
| `norm3` | `LayerNorm` (with affine) | -- |
| `ffn` | `Sequential` (Linear, GELU, Linear) | -- |
| `modulation` | `nn.Parameter [1, 6, dim]` | NB-05 |
| `gate` | `GateModule` | NB-05 |

The modulation and gate sub-modules implement the adaLN-Zero conditioning from NB-05. The SelfAttention and CrossAttention are the same classes from NB-04. Everything already covered -- DiTBlock is just the assembly.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 197-213

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 197-213
dim, num_heads, ffn_dim = 1536, 12, 8960

block = DiTBlock(has_image_input=True, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)

# Sub-module inventory (D-03: parameter counts per sub-module)
print("DiTBlock sub-modules and parameter counts:")
print(f"  self_attn.q  (Linear {dim}->{dim}): {sum(p.numel() for p in block.self_attn.q.parameters()):,} params")
print(f"  self_attn.k  (Linear {dim}->{dim}): {sum(p.numel() for p in block.self_attn.k.parameters()):,} params")
print(f"  self_attn.v  (Linear {dim}->{dim}): {sum(p.numel() for p in block.self_attn.v.parameters()):,} params")
print(f"  self_attn.o  (Linear {dim}->{dim}): {sum(p.numel() for p in block.self_attn.o.parameters()):,} params")
print(f"  cross_attn.q (Linear {dim}->{dim}): {sum(p.numel() for p in block.cross_attn.q.parameters()):,} params")
print(f"  cross_attn.k (Linear {dim}->{dim}): {sum(p.numel() for p in block.cross_attn.k.parameters()):,} params")
print(f"  cross_attn.v (Linear {dim}->{dim}): {sum(p.numel() for p in block.cross_attn.v.parameters()):,} params")
print(f"  cross_attn.o (Linear {dim}->{dim}): {sum(p.numel() for p in block.cross_attn.o.parameters()):,} params")
print(f"  cross_attn.k_img (Linear {dim}->{dim}): {sum(p.numel() for p in block.cross_attn.k_img.parameters()):,} params")
print(f"  cross_attn.v_img (Linear {dim}->{dim}): {sum(p.numel() for p in block.cross_attn.v_img.parameters()):,} params")
print(f"  norm1 (LayerNorm, no affine): {sum(p.numel() for p in block.norm1.parameters()):,} params")
print(f"  norm2 (LayerNorm, no affine): {sum(p.numel() for p in block.norm2.parameters()):,} params")
print(f"  norm3 (LayerNorm, with affine): {sum(p.numel() for p in block.norm3.parameters()):,} params")
print(f"  ffn[0] (Linear {dim}->{ffn_dim}): {sum(p.numel() for p in block.ffn[0].parameters()):,} params")
print(f"  ffn[2] (Linear {ffn_dim}->{dim}): {sum(p.numel() for p in block.ffn[2].parameters()):,} params")
print(f"  modulation (nn.Parameter [1,6,{dim}]): {block.modulation.numel():,} params")
total = sum(p.numel() for p in block.parameters())  # [total block parameters]
print(f"  Block total: {total:,}")

## 2. DiTBlock.forward() -- Line-by-Line Walkthrough

The forward pass has four stages:

1. **Six-parameter chunk** from modulation + t_mod (recall NB-05: adaLN-Zero)
2. **Self-attention branch** with adaLN pre-norm + gated residual (recall NB-04: SelfAttention, NB-01: modulate)
3. **Cross-attention branch** with simple residual -- no adaLN, no gate
4. **FFN branch** with adaLN pre-norm + gated residual (recall NB-01: modulate)

The cross-attention branch is intentionally simpler: it does not use adaLN conditioning or a gate. The intuition is that cross-attention adapts to the text context directly -- it does not need to be gated by the time embedding.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 215-231

### Constructing the 3D Frequency Tensor

`DiTBlock.forward` requires a `freqs` tensor of shape `[seq_len, 1, head_dim//2]`. This is different from the simplified single-band freqs used in NB-04.

**Why the full 3-band assembly is required here:** The `rope_apply` inside `SelfAttention` uses `freqs` to rotate query and key vectors. Each head's key/query is split into three segments for temporal (f), height (h), and width (w) rotations. The last dimension of `freqs` must cover ALL three bands: `22 + 21 + 21 = 64 = head_dim//2`. Using only one band (dim=22) raises `RuntimeError: Sizes of tensors must match` inside `rope_apply`.

The assembly below mirrors `WanModel.forward` lines 381-385 exactly.

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 381-385 (WanModel.forward)
# IMPORTANT: DiTBlock needs FULL 3-band freqs -- NOT the simplified single-band from NB-04
head_dim = dim // num_heads  # 128 = 1536 // 12

f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)
# f_freqs: complex tensor [1024, 22]  (temporal band)
# h_freqs: complex tensor [1024, 21]  (height band)
# w_freqs: complex tensor [1024, 21]  (width band)
print(f"f_freqs: {f_freqs.shape}  # [max_len, 22 complex pairs]")
print(f"h_freqs: {h_freqs.shape}  # [max_len, 21 complex pairs]")
print(f"w_freqs: {w_freqs.shape}  # [max_len, 21 complex pairs]")

# Grid dims after patchify with stride (1,2,2): seq_len = F*H*W
F, H, W = 4, 4, 4  # -> seq_len = 4*4*4 = 64

freqs = torch.cat([
    f_freqs[:F].view(F, 1, 1, -1).expand(F, H, W, -1),  # [F, H, W, 22] temporal band
    h_freqs[:H].view(1, H, 1, -1).expand(F, H, W, -1),  # [F, H, W, 21] height band
    w_freqs[:W].view(1, 1, W, -1).expand(F, H, W, -1),  # [F, H, W, 21] width band
], dim=-1).reshape(F * H * W, 1, -1)                    # [64, 1, 64]  (22+21+21=64=head_dim//2)

assert freqs.shape == torch.Size([F * H * W, 1, head_dim // 2]), \
    f"Expected ({F*H*W}, 1, {head_dim//2}), got {freqs.shape}"
print(f"\nfreqs: {freqs.shape}  # [seq_len, 1, head_dim//2] = [64, 1, 64]")
print(f"Last dim = 22+21+21 = {freqs.shape[-1]} = head_dim//2 = {head_dim//2}")

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 215-231
block.eval()
B, S = 1, F * H * W   # S = 64
x       = torch.randn(B, S, dim)          # [B, S, dim]   -- video tokens
context = torch.randn(B, 277, dim)        # [B, 277, dim] -- 257 CLIP + 20 text tokens
t_mod   = torch.randn(B, 6, dim)          # [B, 6, dim]   -- time modulation from WanModel.time_projection

with torch.no_grad():
    out = block(x, context, t_mod, freqs)  # [B, S, dim]

assert out.shape == torch.Size([B, S, dim]), \
    f"Expected ({B}, {S}, {dim}), got {out.shape}"
print(f"Input:  x {x.shape}")
print(f"Output: {out.shape}")
print("Shape unchanged through DiTBlock -- block transforms features, not layout.")

### Annotated Forward Pass

Below is the `DiTBlock.forward` logic annotated line-by-line. Each stage maps to a concept from a prior notebook.

**Stage 1 -- Six-parameter chunk** (recall NB-05: adaLN-Zero):
- `self.modulation` shape: `[1, 6, dim]` -- learned per-block offset
- `t_mod` shape: `[B, 6, dim]` -- global time conditioning from `WanModel.time_projection`
- Their sum is chunked into six `[B, 1, dim]` tensors: three for self-attention, three for FFN

**Stage 2 -- Self-attention branch** (recall NB-04: SelfAttention, NB-01: modulate):
- `modulate(norm1(x), shift_msa, scale_msa)` applies adaLN pre-norm
- `gate(x, gate_msa, self_attn_output)` applies the gated residual from NB-05

**Stage 3 -- Cross-attention branch** (recall NB-04: CrossAttention):
- Simple residual: `x = x + cross_attn(norm3(x), context)`
- No adaLN conditioning, no gate -- cross-attention adapts to context directly

**Stage 4 -- FFN branch** (recall NB-01: modulate):
- Same adaLN pattern as Stage 2, but using the `mlp` parameters

**Source:** `diffsynth/models/wan_video_dit.py`, lines 219-231

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 219-231
# Annotated DiTBlock.forward line-by-line

# Stage 1: Extract six modulation parameters (recall NB-05: adaLN-Zero)
#   modulation: [1, 6, dim]  (learned per-block offset -- unique to this block)
#   t_mod:      [B, 6, dim]  (global time conditioning from WanModel.time_projection)
#   chunk(6, dim=1) splits [B, 6, dim] into six [B, 1, dim] tensors
shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
    block.modulation.to(dtype=t_mod.dtype, device=t_mod.device) + t_mod
).chunk(6, dim=1)  # six [B, 1, dim] tensors
print(f"Stage 1 -- six parameters from modulation+t_mod chunk:")
for name, val in zip(
    ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"],
    [shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp]
):
    print(f"  {name}: {val.shape}")

# Stage 2: Self-attention branch (recall NB-04: SelfAttention, NB-01: modulate)
x_reset = torch.randn(B, S, dim)   # [B, S, dim] -- fresh x for clear demonstration
input_x = modulate(block.norm1(x_reset), shift_msa, scale_msa)  # [B, S, dim] adaLN pre-norm
with torch.no_grad():
    x_after_sa = block.gate(x_reset, gate_msa, block.self_attn(input_x, freqs))  # [B, S, dim]
print(f"\nStage 2 -- After self-attention + gated residual: {x_after_sa.shape}")

# Stage 3: Cross-attention branch (recall NB-04: CrossAttention) -- simple residual, no gate
with torch.no_grad():
    x_after_ca = x_after_sa + block.cross_attn(block.norm3(x_after_sa), context)  # [B, S, dim]
print(f"Stage 3 -- After cross-attention + simple residual: {x_after_ca.shape}")

# Stage 4: FFN branch (recall NB-01: modulate)
input_x = modulate(block.norm2(x_after_ca), shift_mlp, scale_mlp)  # [B, S, dim] adaLN pre-norm
with torch.no_grad():
    x_final = block.gate(x_after_ca, gate_mlp, block.ffn(input_x))  # [B, S, dim]
print(f"Stage 4 -- After FFN + gated residual: {x_final.shape}")

assert x_final.shape == torch.Size([B, S, dim])
print(f"\nAnnotated forward complete. Shape preserved: {x_final.shape}")

## 3. Parameter Count Breakdown and LoRA Targets

LoRA (Low-Rank Adaptation) fine-tunes a model by learning low-rank updates to selected weight matrices, keeping the original weights frozen. The modules chosen as LoRA targets should be:
1. **Task-specific** -- layers that encode the model's representation of the task, where fine-tuning is most effective
2. **High-dimensional** -- where rank decomposition provides the greatest parameter savings

In `wan_video_dit.py`'s default training config (`train_lora_cached.py` line 292: `--lora_target_modules "q,k,v,o,ffn.0,ffn.2"`), the LoRA targets are:

| Target | Rationale |
|--------|----------|
| `self_attn.q`, `self_attn.k`, `self_attn.v`, `self_attn.o` | Core attention projections -- encode how the model relates video tokens to each other; task-specific spatial-temporal patterns live here |
| `cross_attn.q`, `cross_attn.k`, `cross_attn.v`, `cross_attn.o` | Text conditioning projections -- adapt how text descriptions guide video generation |
| `ffn.0`, `ffn.2` | FFN is the block's "memory" -- high-dimensional feature transform (1536->8960->1536) that stores task-specific representations |

**Not LoRA targets:** `cross_attn.k_img`, `cross_attn.v_img` (image reference path, only active with `has_image_input=True`). These are excluded because the default training uses text+control conditioning, not a reference image.

**Source:** `diffsynth/models/wan_video_dit.py`, line 198; `train_lora_cached.py`, line 292

In [ ]:
# Per-block LoRA target parameter counts (verified: RESEARCH.md Parameter Count Table)
lora_targets = {
    "self_attn.q":  sum(p.numel() for p in block.self_attn.q.parameters()),   # 2,360,832
    "self_attn.k":  sum(p.numel() for p in block.self_attn.k.parameters()),   # 2,360,832
    "self_attn.v":  sum(p.numel() for p in block.self_attn.v.parameters()),   # 2,360,832
    "self_attn.o":  sum(p.numel() for p in block.self_attn.o.parameters()),   # 2,360,832
    "cross_attn.q": sum(p.numel() for p in block.cross_attn.q.parameters()), # 2,360,832
    "cross_attn.k": sum(p.numel() for p in block.cross_attn.k.parameters()), # 2,360,832
    "cross_attn.v": sum(p.numel() for p in block.cross_attn.v.parameters()), # 2,360,832
    "cross_attn.o": sum(p.numel() for p in block.cross_attn.o.parameters()), # 2,360,832
    "ffn.0":        sum(p.numel() for p in block.ffn[0].parameters()),        # 13,771,520
    "ffn.2":        sum(p.numel() for p in block.ffn[2].parameters()),        # 13,764,096
}

block_total = sum(p.numel() for p in block.parameters())  # 51,163,904
lora_total_per_block = sum(lora_targets.values())         # 46,422,272

print("Per-block LoRA target parameter counts:")
for name, count in lora_targets.items():
    pct = count / block_total * 100
    print(f"  {name:<20}: {count:>12,}  ({pct:.1f}%)")
print(f"  {'---':<20}  {'---':>12}")
print(f"  {'LoRA total':<20}: {lora_total_per_block:>12,}  ({lora_total_per_block/block_total*100:.1f}%)")
print(f"  {'Block total':<20}: {block_total:>12,}  (100.0%)")
print()
print("Note: cross_attn.k_img and cross_attn.v_img are NOT LoRA targets")
print("      (default --lora_target_modules 'q,k,v,o,ffn.0,ffn.2' from train_lora_cached.py)")

In [ ]:
# Full-model LoRA scaling: per-block count x 30 blocks (D-04)
num_layers = 30  # WanModel production config

print(f"Full-model LoRA scaling ({num_layers} DiT blocks):")
print(f"  Per-block LoRA params:  {lora_total_per_block:,}")
print(f"  Number of blocks:       {num_layers}")
print(f"  Total LoRA params:      {lora_total_per_block * num_layers:,}")
print()
print(f"30-block LoRA total: {lora_total_per_block * num_layers:,}")
# Verify expected value from RESEARCH.md
assert lora_total_per_block * num_layers == 1_392_668_160, \
    f"Expected 1,392,668,160 but got {lora_total_per_block * num_layers:,}"
print(f"Verified: 30-block LoRA total = 1,392,668,160 params")

## Summary

### Key Takeaways
- **DiTBlock composition**: six sub-modules (`self_attn`, `cross_attn`, `norm1/2/3`, `ffn`, `modulation`, `gate`) assembled in `__init__`; `forward` applies adaLN conditioning twice (before self-attn and before FFN) with a **simple residual** for cross-attn (no adaLN, no gate)
- **freqs construction**: `DiTBlock.forward` requires full 3-band freqs `[seq, 1, head_dim//2]` assembled from `precompute_freqs_cis_3d`; the single-band shortcut from NB-04 raises `RuntimeError` inside `rope_apply`
- **LoRA targets**: `q`, `k`, `v`, `o` in both `self_attn` and `cross_attn`, plus `ffn.0` and `ffn.2` -- **90.7% of block parameters** per block, ~1.39B params across the full 30-block model

### Source References
| Symbol | Location |
|--------|----------|
| `GateModule.forward` | `diffsynth/models/wan_video_dit.py`, line 190 |
| `DiTBlock.__init__` | `diffsynth/models/wan_video_dit.py`, line 198 |
| `DiTBlock.forward` | `diffsynth/models/wan_video_dit.py`, line 215 |
| freqs assembly | `diffsynth/models/wan_video_dit.py`, lines 381-385 |

## Exercises

### Exercise 1 -- Remove cross-attention
Set `has_image_input=False` when creating `DiTBlock`. Rerun the forward pass with a smaller context tensor (`context = torch.randn(B, 20, dim)`). What changes in the parameter count? (Hint: `k_img` and `v_img` disappear from `cross_attn`. Compare the block totals.)

### Exercise 2 -- Inspect gate values
After the annotated forward pass, the variable `gate_msa` has shape `[B, 1, dim]`. Print its mean and std. Is it near zero? Now manually compute what would happen if `gate_msa` were exactly zero: the self-attention output would be zeroed out and the residual stream `x` would pass unchanged. Verify this by running `block.gate(x_reset, torch.zeros_like(gate_msa), torch.randn(B, S, dim))` and checking the output equals `x_reset`.

### Exercise 3 -- Scale LoRA to different ranks
The LoRA target for `self_attn.q` has 2,360,832 parameters (`dim*dim = 1536*1536`). If LoRA uses rank `r=16`, the adapter uses two low-rank matrices: `A` of shape `(r, in_features)` and `B` of shape `(out_features, r)`, for `r * (in_features + out_features)` total parameters. For a square projection (`in=out=1536`): `16 * (1536 + 1536) = 49,152` parameters. Calculate the compression ratio. Repeat for `ffn.0` (`1536->8960`, adapter size = `r * (in_features + out_features) = 16 * (1536 + 8960) = 167,936`).